# Benchmark — Detección de vehículos y estimación de distancias
Mide el tiempo de procesamiento y consumo de recursos del pipeline completo (YOLO + estimación de distancias) para comparar PC vs Raspberry Pi.

In [1]:
import sys
import os
import time
import cv2
import numpy as np
import psutil
import statistics
import platform
import torch
import pynvml


# Ir a la raíz del proyecto para que los paths relativos de las clases funcionen
ruta_proyecto = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(ruta_proyecto)
sys.path.insert(0, os.path.join(ruta_proyecto, 'src'))

from detection.yolo_detection import YoloDetection
from distanceEstimation.Distance_Estimation import DistanceEstimation

# GPU monitoring — intenta pynvml, si falla usa torch.cuda
GPU_DISPONIBLE = False
GPU_MODO       = None
gpu_handle     = None
gpu_name       = "N/A"

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    try:
        pynvml.nvmlInit()
        gpu_handle     = pynvml.nvmlDeviceGetHandleByIndex(0)
        GPU_DISPONIBLE = True
        GPU_MODO       = "pynvml"
    except Exception:
        print("Error Inicializando pynvml")        


if GPU_DISPONIBLE:
    print(f"GPU detectada ({GPU_MODO}) : {gpu_name}")
else:
    print("GPU no disponible — solo se mide CPU/RAM")

print(f"Plataforma : {platform.machine()} — {platform.system()} {platform.release()}")
print(f"Python     : {platform.python_version()}")
print(f"Directorio : {os.getcwd()}")

GPU detectada (pynvml) : NVIDIA GeForce RTX 4070 Laptop GPU
Plataforma : AMD64 — Windows 10
Python     : 3.11.9
Directorio : c:\Users\nicoc\Downloads\Tesis NNs\TesisLab\Deteccion-de-vehiculos-y-estimacion-de-distancias-con-YOLOv8


In [ ]:
# ── Cargar modelo y video ──────────────────────────────────────────────────
rutaModelo = os.path.join(ruta_proyecto, 'models', 'yolov8n.pt')
rutaVideo  = os.path.join(ruta_proyecto, 'data', 'samples', 'Evaluation.mp4')

detector = YoloDetection(rutaModelo)

cap = cv2.VideoCapture(rutaVideo)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_video    = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames a {fps_video:.1f} FPS")

# Warm-up: 20 inferencias previas para estabilizar YOLO antes de medir
N_WARMUP = 20
for _ in range(N_WARMUP):
    ret, frame_warmup = cap.read()
    if ret:
        detector.detectAndParse(frame_warmup)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
print(f"Warm-up completado ({N_WARMUP} frames)")

Video: 617 frames a 30.0 FPS

0: 384x640 8 cars, 93.3ms
Speed: 5.7ms preprocess, 93.3ms inference, 23.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 8.8ms
Speed: 2.1ms preprocess, 8.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 7.7ms
Speed: 1.8ms preprocess, 7.7ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 9.0ms
Speed: 1.6ms preprocess, 9.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 8.5ms
Speed: 1.6ms preprocess, 8.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 11.5ms
Speed: 1.8ms preprocess, 11.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 7.3ms
Speed: 1.7ms preprocess, 7.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 8.7ms
Speed: 1.3ms preprocess, 8.7ms inference, 1.4ms postprocess per image at shape (1, 

In [3]:
# ── Loop de benchmark ─────────────────────────────────────────────────────
tiempos_deteccion = []
tiempos_distancia = []
tiempos_umbral    = []
tiempos_total     = []
uso_cpu           = []
uso_ram_mb        = []
uso_gpu_pct       = []
uso_gpu_mem_mb    = []

proceso = psutil.Process(os.getpid())
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    t_inicio = time.perf_counter()

    t1 = time.perf_counter()
    detecciones = detector.detectAndParse(frame)
    t2 = time.perf_counter()

    z_ref, obj_min, distancias_vec = None, None, []
    try:
        z_ref, obj_min, distancias_vec = DistanceEstimation.distanciasIntervehiculares(detecciones)
    except Exception:
        pass
    t3 = time.perf_counter()

    try:
        if z_ref is not None:
            DistanceEstimation.clasificacionDeDistancia(z_ref)
        for d_AB, _ in distancias_vec:
            DistanceEstimation.clasificacionDeDistancia(d_AB)
    except Exception:
        pass
    t4 = time.perf_counter()

    tiempos_deteccion.append(t2 - t1)
    tiempos_distancia.append(t3 - t2)
    tiempos_umbral.append(t4 - t3)
    tiempos_total.append(t4 - t_inicio)
    uso_cpu.append(psutil.cpu_percent(interval=None))
    uso_ram_mb.append(proceso.memory_info().rss / 1024**2)

    if GPU_DISPONIBLE:
        if GPU_MODO == "pynvml":
            uso_gpu_pct.append(pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu)
            uso_gpu_mem_mb.append(pynvml.nvmlDeviceGetMemoryInfo(gpu_handle).used / 1024**2)
        else:
            uso_gpu_pct.append(torch.cuda.utilization(0))
            uso_gpu_mem_mb.append(torch.cuda.memory_allocated(0) / 1024**2)

cap.release()
print(f"Frames procesados: {len(tiempos_total)}")

0: 384x640 8 cars, 12.2ms
Speed: 2.0ms preprocess, 12.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
8.194365668374196 {'clase_id': 2, 'H_px': 286.64886474609375, 'H_mean': 1.55, 'bbox': (413.8773193359375, 508.60528564453125, 810.4696655273438, 795.254150390625), 'conf': 0.8616598844528198}

0: 384x640 7 cars, 9.0ms
Speed: 2.0ms preprocess, 9.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
8.202156493902 {'clase_id': 2, 'H_px': 281.5560607910156, 'H_mean': 1.55, 'bbox': (935.3944702148438, 504.5398864746094, 1272.432373046875, 786.095947265625), 'conf': 0.8515487313270569}

0: 384x640 9 cars, 8.3ms
Speed: 1.7ms preprocess, 8.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Vehiculo más cercano
8.225094009319228 {'clase_id': 2, 'H_px': 280.7650451660156, 'H_mean': 1.55, 'bbox': (934.6551513671875, 502.7654113769531, 1272.6187744140625, 783.5304565429688), 'conf': 0.8468301296234131}

In [4]:
# ── Resultados ────────────────────────────────────────────────────────────
# Cambiá este valor según dónde corrás el notebook
PLATAFORMA = "PC"   # "PC" o "Raspberry Pi"

def resumen(nombre, datos_s):
    datos_ms = np.array(datos_s) * 1000
    print(f"  {nombre}")
    print(f"    Media   : {datos_ms.mean():.3f} ms")
    print(f"    Mediana : {np.median(datos_ms):.3f} ms")
    print(f"    P95     : {np.percentile(datos_ms, 95):.3f} ms")
    print(f"    P99     : {np.percentile(datos_ms, 99):.3f} ms")
    print(f"    Mín     : {datos_ms.min():.3f} ms")
    print(f"    Máx     : {datos_ms.max():.3f} ms")

n = len(tiempos_total)
fps_real = 1 / np.mean(tiempos_total)

print(f"========== BENCHMARK — {PLATAFORMA} ==========")
print(f"Frames procesados : {n}")
print()
resumen("1. Detección YOLO",            tiempos_deteccion)
print()
resumen("2. Estimación de distancias",   tiempos_distancia)
print()
resumen("3. Clasificación por umbrales", tiempos_umbral)
print()
resumen("Pipeline completo (1+2+3)",     tiempos_total)
print(f"    FPS estimados : {fps_real:.2f}")
print()
print(f"  CPU / RAM")
print(f"    CPU media : {np.mean(uso_cpu):.1f}%")
print(f"    RAM media : {np.mean(uso_ram_mb):.1f} MB")
print(f"    RAM pico  : {np.max(uso_ram_mb):.1f} MB")

if GPU_DISPONIBLE and uso_gpu_pct:
    print()
    print(f"  GPU ({gpu_name})")
    print(f"    Utilización media : {np.mean(uso_gpu_pct):.1f}%")
    print(f"    Utilización pico  : {np.max(uso_gpu_pct):.1f}%")
    print(f"    VRAM media        : {np.mean(uso_gpu_mem_mb):.1f} MB")
    print(f"    VRAM pico         : {np.max(uso_gpu_mem_mb):.1f} MB")

========== BENCHMARK — PC ==========
Frames procesados : 617

  1. Detección YOLO
    Media   : 14.610 ms
    Mediana : 13.991 ms
    P95     : 18.052 ms
    P99     : 20.954 ms
    Mín     : 11.611 ms
    Máx     : 115.637 ms

  2. Estimación de distancias
    Media   : 0.909 ms
    Mediana : 0.785 ms
    P95     : 1.317 ms
    P99     : 1.631 ms
    Mín     : 0.553 ms
    Máx     : 38.817 ms

  3. Clasificación por umbrales
    Media   : 0.003 ms
    Mediana : 0.003 ms
    P95     : 0.005 ms
    P99     : 0.006 ms
    Mín     : 0.002 ms
    Máx     : 0.009 ms

  Pipeline completo (1+2+3)
    Media   : 15.522 ms
    Mediana : 14.826 ms
    P95     : 18.999 ms
    P99     : 21.974 ms
    Mín     : 12.384 ms
    Máx     : 116.561 ms
    FPS estimados : 64.42

  CPU / RAM
    CPU media : 35.3%
    RAM media : 1477.8 MB
    RAM pico  : 1481.5 MB

  GPU (NVIDIA GeForce RTX 4070 Laptop GPU)
    Utilización media : 26.0%
    Utilización pico  : 49.0%
    VRAM media        : 2084.3 MB
    VRA